# POPULATION INSIGHTS
## India & World Population Analysis and Visualization
**Course:** Data Visualization using Python  
**Core Libraries:** Pandas · NumPy · Matplotlib  

---
### Visualizations in this Notebook
| # | Chart Type | Matplotlib Method |
|---|---|---|
| 1 | Multi-Year Population Line Chart | `plt.plot()` |
| 2 | Country Share Pie Chart | `plt.pie()` |
| 3 | Indian States Horizontal Bar Chart | `plt.barh()` |
| 4 | Age-Sex Demographic Population Pyramid | back-to-back `plt.barh()` |
| 5 | Population Size vs. Density Scatter Plot | `plt.scatter()` |
| 6 | Growth Rates Heatmap | `plt.imshow()` |
| 7 | Cumulative Growth Comparison Bar Chart | `plt.bar()` |
| 8 | Selected-Country Population Distribution Histogram | `plt.hist()` |

> **Data:** This notebook loads all five CSV datasets directly from the project’s public GitHub repository. No manual CSV upload is needed in Google Colab.


## Cell 0 — Install Packages & Set Up

In [ ]:
# Install required packages (runs automatically on Colab/Jupyter)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'matplotlib', 'numpy'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 90
plt.rcParams['savefig.bbox'] = 'tight'

# Choose a style that exists in all environments
STYLE = 'seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'ggplot'

print('Libraries ready!')
print('Matplotlib style used:', STYLE)

## Step 1 — Load the Project CSV Datasets
The notebook downloads and uses all five CSV files from the project repository. In Google Colab, run this cell while connected to the internet; the files do not need to be uploaded manually. If the repository is already cloned or the CSVs are in the current folder, the local copies are used instead.


In [ ]:
# Load every CSV dataset from the repository (or from the current/local data folder).
from pathlib import Path

REPO_RAW = 'https://raw.githubusercontent.com/Dev-Neo156/Population-Data-visualization-using-python/main'
CSV_FILES = [
    'world_population_historical.csv',
    'world_population_wide.csv',
    'india_census_states.csv',
    'india_age_sex_structure.csv',
    'global_density_growth.csv',
]

# Prefer local files when present; otherwise fetch the public repository copies.
SEARCH_DIRS = [Path('.'), Path('data'), Path('/content'), Path('/content/data')]
def find_local_csv(filename):
    for folder in SEARCH_DIRS:
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    return None

def load_project_csv(filename):
    local_path = find_local_csv(filename)
    if local_path is not None:
        print(f'Loaded local file: {local_path}')
        return pd.read_csv(local_path)
    url = f'{REPO_RAW}/{filename}'
    print(f'Downloading: {filename}')
    return pd.read_csv(url)

csv_data = {name: load_project_csv(name) for name in CSV_FILES}

# Keep all source tables available, including the wide-format companion dataset.
df_world = csv_data['world_population_historical.csv'].rename(
    columns={'Country Name': 'Country'}
)
df_world_wide = csv_data['world_population_wide.csv'].rename(
    columns={'Country Name': 'Country'}
)
df_states = csv_data['india_census_states.csv']
df_pyramid = csv_data['india_age_sex_structure.csv'].rename(columns={
    'Male_Millions': 'Male_M',
    'Female_Millions': 'Female_M',
})
df_density = csv_data['global_density_growth.csv']

# Basic type cleanup makes the plotting and calculation cells work consistently.
df_world['Year'] = pd.to_numeric(df_world['Year'], errors='raise').astype(int)
df_world['Population'] = pd.to_numeric(df_world['Population'], errors='raise')
for col in ['Population_2011']:
    df_states[col] = pd.to_numeric(df_states[col], errors='raise')
for col in ['Male_M', 'Female_M']:
    df_pyramid[col] = pd.to_numeric(df_pyramid[col], errors='raise')

print('CSV datasets ready:')
for filename, frame in csv_data.items():
    print(f'  {filename}: {frame.shape[0]} rows × {frame.shape[1]} columns')
print('World historical columns:', df_world.columns.tolist())


---
## Visualization 1 — Line Chart (`plt.plot()`)
### Multi-Year Population Growth: India, China, and the United States (1960 – 2024)
**Objective:** Track long-run population trajectories and identify the demographic crossover moment.

In [ ]:
plt.style.use(STYLE)
fig, ax = plt.subplots(figsize=(9, 5))

colors  = {'India': '#FF9933', 'China': '#DE2910', 'United States': '#0A3161'}
markers = {'India': 'o',       'China': 's',       'United States': '^'}

for country in ['India', 'China', 'United States']:
    d = df_world[df_world['Country'] == country].sort_values('Year')
    ax.plot(d['Year'], d['Population'] / 1e9,
            label=country, color=colors[country], linewidth=2.8,
            marker=markers[country], markevery=5, markersize=6, alpha=0.9)

# Find the crossover year from the data (first year India's population exceeds China's)
india_s = df_world[df_world['Country'] == 'India'].set_index('Year')['Population']
china_s = df_world[df_world['Country'] == 'China'].set_index('Year')['Population']
cross_year  = int((india_s - china_s)[lambda d: d > 0].index.min())
india_cross = india_s[cross_year] / 1e9

# Annotate the crossover
ax.scatter([cross_year], [india_cross], color='#D81B60', s=130, zorder=5)
ax.annotate(f'{cross_year}: India\nsurpasses China',
            xy=(cross_year, india_cross), xytext=(2005, india_cross + 0.12),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5),
            fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', fc='#FFF8E1', ec='#FFA000', alpha=0.95))

ax.set_title('Population Trajectory: India, China & USA (1960 – 2024)', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Population (Billions)', fontsize=11)
ax.set_xlim(1958, 2026)
ax.set_ylim(0, 1.6)
ax.set_xticks(np.arange(1960, 2025, 10))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.1f}B'))
ax.legend(title='Country', fontsize=10, title_fontsize=10)
ax.grid(True, linestyle='--', alpha=0.6)
plt.figtext(0.12, -0.02,
    'Source: World Bank (SP.POP.TOTL) & UN WPP 2024. Years between benchmark points are interpolated.',
    fontsize=8, color='#555', style='italic')
plt.tight_layout()
plt.show()

print(f'Observation: In this dataset India overtakes China in {cross_year} (official UN estimates say about 2023; years between benchmark points are interpolated). '
      'China has entered near-zero growth while India maintains replacement-level fertility.')

---
## Visualization 2 — Pie Chart (`plt.pie()`)
### Proportional Share of Combined Population — Top 10 Nations (2024)
**Objective:** Show each country's relative weight in a high-population cohort.

In [ ]:
plt.style.use(STYLE)
pie_countries = ['India','China','United States','Indonesia','Pakistan',
                  'Nigeria','Brazil','Bangladesh','Russia','Japan']
pie_data = (df_world[(df_world['Country'].isin(pie_countries)) & (df_world['Year']==2024)]
            .sort_values('Population', ascending=False))

labels_list = pie_data['Country'].tolist()
pops        = pie_data['Population'].to_numpy()
shares      = np.round(np.divide(pops, pops.sum()) * 100, 2)
explode     = [0.08, 0.05] + [0.0] * (len(labels_list) - 2)

palette = ['#FF9933','#DE2910','#1976D2','#388E3C','#7B1FA2',
           '#F57C00','#0097A7','#5D4037','#689F38','#455A64']

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    pops, explode=explode, colors=palette, startangle=140,
    autopct='%1.1f%%', pctdistance=0.78,
    wedgeprops=dict(width=0.85, edgecolor='white', linewidth=1.8))

for at in autotexts:
    at.set_color('white'); at.set_fontsize(9.5); at.set_weight('bold')

legend_labels = [f'{c}  {p/1e6:.0f}M  ({s:.1f}%)'
                 for c, p, s in zip(labels_list, pops, shares)]
ax.legend(wedges, legend_labels, title=f'Total combined: {pops.sum()/1e9:.2f}B',
          loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=9)

ax.set_title('Proportional Distribution of Combined Population (2024)\nTop 10 High-Population Nations',
             fontsize=14, fontweight='bold', pad=16)
plt.figtext(0.08, -0.02,
    'Source: World Bank & UN WPP 2024 Estimates.',
    fontsize=8, color='#555', style='italic')
plt.tight_layout()
plt.show()

top2 = shares[:2].sum()
print(f'Observation: India + China together account for ~{top2:.1f}% of the combined population of these 10 nations.')

---
## Visualization 3 — Horizontal Bar Chart (`plt.barh()`)
### Ranking Indian States by Population — Census of India 2011
**Objective:** Identify the most and least populous Indian states.

In [ ]:
plt.style.use(STYLE)
top15  = df_states.sort_values('Population_2011', ascending=True).tail(15)
states = top15['State'].tolist()
pops_m = np.divide(top15['Population_2011'].to_numpy(np.float64), 1e6)

colors_bar = ['#4A90E2'] * len(states)
colors_bar[-1] = '#D9534F'  # UP — largest
colors_bar[-2] = '#F0AD4E'  # Maharashtra
colors_bar[-3] = '#5CB85C'  # Bihar

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(states, pops_m, color=colors_bar, height=0.65,
               edgecolor='black', linewidth=0.5, alpha=0.9)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 2, bar.get_y() + bar.get_height()/2,
            f'{w:.1f}M ({w/10:.1f} Cr)',
            va='center', ha='left', fontsize=9, fontweight='bold', color='#222')

ax.set_title('Ranking Indian States by Population\nCensus of India 2011 — Top 15',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Population (Millions)', fontsize=11)
ax.set_xlim(0, 230)
ax.grid(axis='x', linestyle='--', alpha=0.6)
plt.figtext(0.08, -0.02,
    'Source: Office of the Registrar General & Census Commissioner, India (Census 2011).',
    fontsize=8, color='#555', style='italic')
plt.tight_layout()
plt.show()

up_pop      = df_states['Population_2011'].max() / 1e6
brazil_2010 = df_world[(df_world['Country'] == 'Brazil') & (df_world['Year'] == 2010)]['Population'].values[0] / 1e6
print(f'Observation: Uttar Pradesh alone has {up_pop:.1f}M people — about 16.5% of India (Census 2011). '
      f'That is roughly the size of Brazil in 2010 ({brazil_2010:.1f}M).')

---
## Visualization 4 — Population Pyramid (Back-to-Back `plt.barh()`)
### India Demographic Structure: Male vs. Female Age Cohorts
**Objective:** Visualize India's age-sex structure and demographic dividend.

In [ ]:
plt.style.use(STYLE)
age_groups_list = df_pyramid['Age_Group'].tolist()
males_neg       = -df_pyramid['Male_M'].to_numpy()   # negative = extends left
females_pos     =  df_pyramid['Female_M'].to_numpy()
y_pos           = np.arange(len(age_groups_list))

fig, ax = plt.subplots(figsize=(9, 6.5))
rm = ax.barh(y_pos, males_neg,   height=0.72, color='#1E88E5', label='Male',
             edgecolor='black', linewidth=0.4, alpha=0.9)
rf = ax.barh(y_pos, females_pos, height=0.72, color='#E53935', label='Female',
             edgecolor='black', linewidth=0.4, alpha=0.9)

for r in list(rm) + list(rf):
    val = abs(r.get_width())
    if val > 20:
        sign = 1 if r.get_width() > 0 else -1
        ax.text(r.get_width() - sign * 1.5, r.get_y() + r.get_height()/2,
                f'{val:.0f}M', va='center', ha='right' if sign > 0 else 'left',
                color='white', fontsize=8, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(age_groups_list, fontsize=9.5, fontweight='bold')
ticks = np.arange(-70, 75, 10)
ax.set_xticks(ticks)
ax.set_xticklabels([f'{abs(t)}M' for t in ticks])
ax.set_xlim(-75, 75)
ax.axvline(0, color='black', linewidth=1.2)
ax.set_xlabel('Population (Millions)', fontsize=11)
ax.set_ylabel('Age Cohort', fontsize=11)
ax.set_title('India Demographic Structure: Age-Sex Population Pyramid\nMale (left) vs. Female (right)',
             fontsize=14, fontweight='bold', pad=14)
ax.legend(fontsize=10)
ax.grid(axis='x', linestyle='--', alpha=0.5)

ax.text(-73, 13.8, 'Aging Cohorts (65+)\nSmaller share', fontsize=8.5,
        bbox=dict(boxstyle='round', fc='#FFF3E0', ec='#FFB74D'))
ax.text(-73, 4.5, 'Working-Age Bulge (15-39)\nDemographic Dividend', fontsize=8.5,
        bbox=dict(boxstyle='round', fc='#E8F5E9', ec='#81C784'))

plt.figtext(0.08, -0.02,
    'Source: Office of the Registrar General of India & UN WPP.',
    fontsize=8, color='#555', style='italic')
plt.tight_layout()
plt.show()

total_pyr = (df_pyramid['Male_M'].sum() + df_pyramid['Female_M'].sum()) / 1e3
print('Observation: India has a massive 15-39 working-age bulge — a classic Demographic Dividend structure.')
print(f'(Cohort sizes show the shape of the age structure; they sum to {total_pyr:.2f}B, not exactly the 2024 total.)')

---
## Visualization 5 — Scatter Plot (`plt.scatter()`)
### Population Size vs. Population Density Across Global Regions (2024)
**Objective:** Explore how land area, population size, and density interrelate across regions.

In [ ]:
plt.style.use(STYLE)

region_colors = {
    'South Asia':         '#FF9933',
    'East Asia':          '#DE2910',
    'North America':      '#1E88E5',
    'Southeast Asia':     '#00897B',
    'Sub-Saharan Africa': '#8E24AA',
    'Latin America':      '#43A047',
    'Europe/Asia':        '#78909C',
    'Europe':             '#5C6BC0',
    'Middle East':        '#F4511E',
    'Oceania':            '#00ACC1',
}

pops_m    = np.divide(df_density['Population_2024'].to_numpy(np.float64), 1e6)
densities = df_density['Density_per_km2'].to_numpy()
areas     = df_density['Area_km2'].to_numpy(np.float64)
pt_colors = [region_colors.get(r, '#9E9E9E') for r in df_density['Region']]
bubble_sz = np.sqrt(areas) / 8 + 60

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(pops_m, densities, s=bubble_sz, c=pt_colors,
           alpha=0.82, edgecolors='black', linewidth=1.1, zorder=3)

offsets = {
    'India': (-60, 20), 'China': (-65, -28), 'Bangladesh': (15, -18),
    'Russia': (10, -22), 'United States': (10, 12),
}
for _, row in df_density.iterrows():
    ox, oy = offsets.get(row['Country'], (10, 10))
    ax.annotate(row['Country'], xy=(row['Population_2024']/1e6, row['Density_per_km2']),
                xytext=(ox, oy), textcoords='offset points',
                fontsize=8.5, fontweight='bold', color='#111',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='#CCC', alpha=0.85),
                arrowprops=dict(arrowstyle='-', color='#AAA', lw=0.7))

ax.set_yscale('log')
ax.set_yticks([3, 10, 30, 100, 300, 1000])
ax.get_yaxis().set_major_formatter(mticker.ScalarFormatter())
ax.set_xlim(-40, 1600)
ax.set_xlabel('Total Population (Millions)', fontsize=11)
ax.set_ylabel('Population Density (people / km²) — Log Scale', fontsize=11)
ax.set_title('Global Demographics: Population Size vs. Density (2024)\nBubble size = square root of land area',
             fontsize=14, fontweight='bold', pad=14)
ax.grid(True, which='both', linestyle='--', alpha=0.45)

handles = [plt.Line2D([0],[0], marker='o', color='w',
           markerfacecolor=c, markersize=9, label=r)
           for r, c in region_colors.items()
           if r in df_density['Region'].unique()]
ax.legend(handles=handles, title='Region', loc='lower right', fontsize=8.5, title_fontsize=9)

ax.annotate('Extreme density:\nBangladesh >1180/km²',
            xy=(174.7, 1183), xytext=(350, 700),
            arrowprops=dict(arrowstyle='->', color='#C2185B', lw=1.5),
            fontsize=9, fontweight='bold', color='#C2185B',
            bbox=dict(boxstyle='round,pad=0.4', fc='#FCE4EC', ec='#F06292'))

plt.figtext(0.08, -0.02,
    'Source: UN WPP 2024 & World Bank. Y-axis is log-scaled.',
    fontsize=8, color='#555', style='italic')
plt.tight_layout()
plt.show()

print('Observation: Bangladesh is a density outlier at >1,180/km². Canada and Australia have <5/km² despite massive land areas.')

---
## Visualization 6 — Heatmap (`plt.imshow()`)
### Demographic Transition: Annual Growth Rate Matrix (1980 – 2024)
**Objective:** Show how annual growth rates have changed across countries and decades using color intensity.

In [ ]:
plt.style.use(STYLE)
heat_countries = ['Nigeria','Ethiopia','Pakistan','Egypt',
                   'India','Indonesia','Mexico','Brazil',
                   'United States','China','Germany','Russia','Japan']

growth_cols = ['Growth_1980','Growth_1990','Growth_2000','Growth_2010','Growth_2020','Growth_2024']
year_labels = ['1980','1990','2000','2010','2020','2024']

heat_df = (df_density[df_density['Country'].isin(heat_countries)]
           .sort_values('Growth_2024', ascending=False))
heat_matrix  = heat_df[growth_cols].to_numpy()   # 2-D NumPy array
country_list = heat_df['Country'].tolist()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(heat_matrix, cmap='YlOrRd', aspect='auto', interpolation='nearest')

ax.set_xticks(np.arange(len(year_labels)))
ax.set_yticks(np.arange(len(country_list)))
ax.set_xticklabels(year_labels, fontsize=10.5, fontweight='bold')
ax.set_yticklabels(country_list, fontsize=10.5, fontweight='bold')
ax.set_xlabel('Observation Year', fontsize=11, labelpad=10)
ax.set_ylabel('Country (sorted by 2024 growth rate)', fontsize=11, labelpad=10)

for i in range(len(country_list)):
    for j in range(len(year_labels)):
        v = heat_matrix[i, j]
        ax.text(j, i, f'{v:+.2f}%', ha='center', va='center',
                color='white' if v > 2.0 else 'black', fontsize=9, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.03)
cbar.set_label('Annual Population Growth Rate (%)', fontsize=10, rotation=270, labelpad=16)

ax.set_title('Demographic Transition Heatmap: Annual Growth Rates (1980 – 2024)\nGlobal Fertility Decline & Regional Trajectories',
             fontsize=13, fontweight='bold', pad=14)
plt.figtext(0.08, -0.02,
    'Source: UN Population Division & World Bank. Method: plt.imshow().',
    fontsize=8, color='#555', style='italic')
plt.tight_layout()
plt.show()

print('Observation: Sub-Saharan Africa remains hot (>2.3%). Japan/China/Russia have flipped to negative growth — visible in the white/pale cells.')

---
## Visualization 7 — Cumulative Growth Bar Chart (`plt.bar()`)
### Cumulative Population Growth Comparison (2000 – 2024)
**Objective:** Compare 24-year percentage growth across major economies from the 2000 and 2024 population values.

In [ ]:
plt.style.use(STYLE)
bar_countries = ['Nigeria','Pakistan','India','Bangladesh',
                  'Brazil','United States','China','Russia','Japan']

bpivot  = (df_world[df_world['Country'].isin(bar_countries) & df_world['Year'].isin([2000,2024])]
           .pivot(index='Country', columns='Year', values='Population'))
bp_prev = bpivot[2000].to_numpy(np.float64)
bp_curr = bpivot[2024].to_numpy(np.float64)
growth_rates = (bp_curr - bp_prev) / bp_prev * 100

sort_idx     = np.argsort(growth_rates)[::-1]
sorted_names = [bpivot.index[i] for i in sort_idx]
sorted_rates = growth_rates[sort_idx]

bar_colors = ['#2E7D32' if g >= 30 else '#1976D2' if g > 0 else '#C62828'
              for g in sorted_rates]

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.bar(sorted_names, sorted_rates, color=bar_colors,
              width=0.58, edgecolor='black', linewidth=0.6, alpha=0.9)

for bar in bars:
    h  = bar.get_height()
    yp = h + 1.5 if h >= 0 else h - 3.5
    ax.text(bar.get_x() + bar.get_width()/2, yp, f'{h:+.1f}%',
            ha='center', fontsize=9.5, fontweight='bold', color='#1B1B1B')

mean_gr = np.mean(sorted_rates)   # np.mean() across the array
ax.axhline(0,       color='black', linewidth=1.1)
ax.axhline(mean_gr, color='#E65100', linewidth=1.5, linestyle='--',
           label=f'Group Mean: {mean_gr:.1f}%')

ax.set_title('Cumulative Population Growth (2000 – 2024)\nCalculated with NumPy: np.divide(curr - prev, prev) × 100',
             fontsize=14, fontweight='bold', pad=14)
ax.set_ylabel('Population Change (%)', fontsize=11)
ax.set_ylim(min(sorted_rates) - 8, max(sorted_rates) + 14)
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(rotation=25, ha='right', fontsize=10, fontweight='bold')
ax.legend(fontsize=10)
plt.figtext(0.08, -0.04,
    'Source: World Bank & UN WPP. Nigeria near-doubled; Japan contracted by -2.44%.',
    fontsize=8, color='#555', style='italic')
plt.tight_layout()
plt.show()

print('Observation: Nigeria grew +89.4%. India added 385M people (+36.4%). Japan: -2.44% decline (demographic contraction).')

---
## Visualization 8 — Histogram (`plt.hist()`)
### Population Distribution Across 10 Selected Countries (2024)
**Objective:** Compare country populations among the same 10 countries used in this section. Log-spaced bins make the strongly right-skewed distribution readable; the vertical line marks the median.


In [ ]:
plt.style.use(STYLE)

# Keep the cohort explicit and identical in the histogram.
selected_countries = ['India', 'China', 'United States', 'Indonesia', 'Pakistan',
                      'Nigeria', 'Brazil', 'Bangladesh', 'Russia', 'Japan']
latest_year = int(df_world['Year'].max())
country_rows = df_world[(df_world['Year'] == latest_year) &
                        (df_world['Country'].isin(selected_countries))]
if country_rows['Country'].duplicated().any():
    raise ValueError('Expected exactly one population row per selected country and year.')
missing = sorted(set(selected_countries) - set(country_rows['Country']))
if missing:
    raise ValueError(f'Missing selected countries for {latest_year}: {missing}')

pop_by_country = (country_rows.set_index('Country')['Population']
                  .reindex(selected_countries).dropna() / 1e6)
pop_2024 = pop_by_country.to_numpy(np.float64)
median_pop = np.median(pop_2024)
country_count = len(pop_by_country)

# Equal-width intervals in log space avoid one huge empty range from India and China.
bins = np.geomspace(pop_2024.min(), pop_2024.max(), 7)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(pop_2024, bins=bins, color='#1976D2', edgecolor='white',
        linewidth=1.2, alpha=0.9)
ax.axvline(median_pop, color='#E65100', linewidth=2, linestyle='--',
           label=f'Median: {median_pop:.1f}M')
ax.set_xscale('log')
ax.set_xticks([100, 200, 500, 1000, 1500])
ax.get_xaxis().set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.set_title(f'Population Distribution Across {country_count} Selected Countries ({latest_year})',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Population (Millions, logarithmic scale; equal-width log bins)', fontsize=11)
ax.set_ylabel('Number of Countries', fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(fontsize=10)
plt.figtext(0.08, -0.02,
    'Source: World Bank & UN WPP. Ten selected countries; World aggregate excluded.',
    fontsize=8, color='#555', style='italic')
plt.tight_layout()
plt.show()

print(f'Observation: Among the {country_count} selected countries, the median population is {median_pop:.1f}M. '
      'The World aggregate is excluded; logarithmic bins show both the cluster and the two largest populations.')


---
## Summary of Findings & Evidence-Based Conclusions

| # | Key Finding |
|---|---|
| 1 | **The 2022–23 Demographic Pivot:** India surpassed China as the world's most populous nation (in this dataset in 2022; official estimates say about 2023). By 2024 it is ~1.44B vs ~1.41B. China is in near-zero/negative growth; India maintains replacement-level fertility. |
| 2 | **Sub-National Disparities:** Uttar Pradesh (199.8M) = 16.5% of India alone. The top 5 states house >47% of the population. |
| 3 | **Demographic Dividend:** India's pyramid shows a massive productive bulge in the 15–39 age cohorts — a powerful growth opportunity. |
| 4 | **Extreme Carrying Capacity Gap:** Bangladesh (>1,180 people/km²) vs. Canada (<5 people/km²) — same era, opposite land pressures. |
| 5 | **Global Demographic Transition:** All major economies trended toward lower fertility 1980–2024. Africa (Nigeria, Ethiopia) still leads expansion. Japan/China/Russia entered negative growth. |

---
### Libraries Used Summary
```python
import pandas as pd       # Data ingestion, cleaning, pivot, describe
import numpy as np        # np.divide(), np.mean(), np.geomspace(), np.median()
import matplotlib.pyplot as plt  # plt.plot(), plt.pie(), plt.barh(), plt.scatter(), plt.imshow(), plt.bar(), plt.hist()
```